# Ablation Study

4 conditions x 4 frontier models. Tests contribution of CPG reference and ELM simplification.

In [1]:
import pandas as pd, numpy as np, os, warnings
from statsmodels.stats.proportion import proportion_confint
from scipy import stats
warnings.filterwarnings('ignore')

ABL_DIR = None
for d in ['../results/ablation','results/ablation']:
    if os.path.isdir(d): ABL_DIR = d; break

MODES = ['full','no_cpg','no_simplify','no_cpg_no_simplify']
ML = {'full':'Simplified+CPG','no_cpg':'Simplified only',
      'no_simplify':'Raw JSON+CPG','no_cpg_no_simplify':'Raw JSON only'}

frames = []
for f in sorted(os.listdir(ABL_DIR)):
    if not f.endswith('.csv'): continue
    df = pd.read_csv(os.path.join(ABL_DIR, f))
    for col in ['valid','correct','expected_valid']:
        if col in df.columns: df[col] = df[col].astype(str).str.strip().str.lower()=='true'
    for mode in MODES:
        if f'-{mode}.csv' in f: df['ablation_mode']=mode; break
    frames.append(df)
data = pd.concat(frames, ignore_index=True)
models = sorted(data['model'].unique())
print(f"Loaded: {len(models)} models x {data['ablation_mode'].nunique()} conditions")

Loaded: 4 models x 4 conditions


## Accuracy by Condition

In [2]:
rows = []
for model in models:
    for mode in MODES:
        mdf = data[(data['model']==model)&(data['ablation_mode']==mode)]
        if len(mdf)==0: continue
        n=len(mdf); k=int(mdf['correct'].sum()); acc=k/n
        rows.append({'Model':model,'Condition':ML[mode],'Accuracy':acc})
adf = pd.DataFrame(rows)
pivot = adf.pivot(index='Condition',columns='Model',values='Accuracy')
pivot = pivot[models].reindex([ML[m] for m in MODES])
pivot.style.format('{:.1%}').background_gradient(cmap='RdYlGn',vmin=0.5,vmax=1.0).set_caption(
    'Ablation: Accuracy by Condition and Model')

Model,gpt-oss-120b,gpt-oss-20b,llama-3.3-70b,qwen3-32b
Condition,,,,
Simplified+CPG,87.1%,90.3%,90.3%,83.9%
Simplified only,71.0%,61.3%,58.1%,45.2%
Raw JSON+CPG,77.4%,48.4%,96.8%,83.9%
Raw JSON only,67.7%,71.0%,58.1%,54.8%


## Delta from Full Pipeline

In [3]:
print(f"{'Model':<20s} {'No CPG':>10s} {'No Simplify':>12s} {'Neither':>10s}")
print('-'*55)
for model in models:
    accs = {}
    for mode in MODES:
        sub = data[(data['model']==model)&(data['ablation_mode']==mode)]
        if len(sub)>0: accs[mode] = sub['correct'].mean()
    if 'full' not in accs: continue
    full = accs['full']
    d1=accs.get('no_cpg',0)-full; d2=accs.get('no_simplify',0)-full
    d3=accs.get('no_cpg_no_simplify',0)-full
    print(f"{model:<20s} {d1:>+9.1%} {d2:>+11.1%} {d3:>+9.1%}")
print(f"\nKey finding: CPG removal always hurts. Simplification benefit")
print(f"scales inversely with active parameter count.")

Model                    No CPG  No Simplify    Neither
-------------------------------------------------------
gpt-oss-120b            -16.1%       -9.7%    -19.4%
gpt-oss-20b             -29.0%      -41.9%    -19.4%
llama-3.3-70b           -32.3%       +6.5%    -32.3%
qwen3-32b               -38.7%       +0.0%    -29.0%

Key finding: CPG removal always hurts. Simplification benefit
scales inversely with active parameter count.
